<a href="https://colab.research.google.com/github/udaymehta5/Movie-Recommendation-using-machine-learning/blob/main/movie_recommendation_with_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Movie recommendation using python libraries**

In [24]:
from google.colab import files
files.upload()


Saving movies.csv to movies (1).csv
Saving ratings.csv to ratings (1).csv


{'movies (1).csv': b'movieId,title,genres\n1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy\n2,Jumanji (1995),Adventure|Children|Fantasy\n3,Grumpier Old Men (1995),Comedy|Romance\n4,Waiting to Exhale (1995),Comedy|Drama|Romance\n5,Father of the Bride Part II (1995),Comedy\n',
 'ratings (1).csv': b'userId,movieId,rating,timestamp\n1,1,4.0,964982703\n1,2,5.0,964982703\n2,1,3.5,964982703\n2,3,4.5,964982703\n3,2,4.0,964982703\n3,4,2.5,964982703\n4,3,3.0,964982703\n4,5,4.5,964982703\n'}

# **Import the libraries**

In [25]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


# **Load the dataset**

In [26]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

movies.head()


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [27]:
ratings.head()


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,2,5.0,964982703
2,2,1,3.5,964982703
3,2,3,4.5,964982703
4,3,2,4.0,964982703


# **Understand the data**

In [28]:
print(movies.info())
print(ratings.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  5 non-null      int64 
 1   title    5 non-null      object
 2   genres   5 non-null      object
dtypes: int64(1), object(2)
memory usage: 252.0+ bytes
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   userId     8 non-null      int64  
 1   movieId    8 non-null      int64  
 2   rating     8 non-null      float64
 3   timestamp  8 non-null      int64  
dtypes: float64(1), int64(3)
memory usage: 388.0 bytes
None


# **User movie matrix**

In [29]:
user_movie_matrix = ratings.pivot(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

user_movie_matrix.head()


movieId,1,2,3,4,5
userId,,,,,
1,4.0,5.0,0.0,0.0,0.0
2,3.5,0.0,4.5,0.0,0.0
3,0.0,4.0,0.0,2.5,0.0
4,0.0,0.0,3.0,0.0,4.5


# **Similarities**

In [30]:
movie_similarity = cosine_similarity(user_movie_matrix.T)

In [31]:
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=user_movie_matrix.columns,
    columns=user_movie_matrix.columns
)

movie_similarity_df.head()


movieId,1,2,3,4,5
movieId,,,,,
1,1.000000,0.587664,0.547909,0.000000,0.0000
2,0.587664,1.000000,0.000000,0.624695,0.0000
3,0.547909,0.000000,1.000000,0.000000,0.5547
4,0.000000,0.624695,0.000000,1.000000,0.0000
5,0.000000,0.000000,0.554700,0.000000,1.0000


# **Recommendation Function**

In [32]:
def recommend_movies(movie_title, num_recommendations=5):
    # Get movieId from title
    movie_id = movies[movies['title'] == movie_title]['movieId'].values

    if len(movie_id) == 0:
        return "Movie not found"

    movie_id = movie_id[0]

    # Get similarity scores
    similarity_scores = movie_similarity_df[movie_id]

    # Sort movies by similarity
    similar_movies = similarity_scores.sort_values(ascending=False)[1:num_recommendations+1]

    # Get movie titles
    recommended_titles = movies[movies['movieId'].isin(similar_movies.index)]['title']

    return recommended_titles.tolist()


In [33]:
recommend_movies("Toy Story (1995)", 5)


['Jumanji (1995)',
 'Grumpier Old Men (1995)',
 'Waiting to Exhale (1995)',
 'Father of the Bride Part II (1995)']